In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

In [7]:
model_url = "https://tfhub.dev/sayakpaul/vit_b16_fe/1"
vit_model = hub.KerasLayer(model_url,trainable=True)

In [4]:
train_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Train'
test_path = '/content/drive/Othercomputers/My Laptop/RoadImagesDataSet/Test'

In [8]:
# Define image size and batch size
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# Data generators for loading and augmenting the data
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255, validation_split=0.2,
)

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='validation'
)


Found 5043 images belonging to 7 classes.
Found 1257 images belonging to 7 classes.


In [9]:
# Build the model
inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
x = vit_model(inputs)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.5)(x)
outputs = tf.keras.layers.Dense(7, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [10]:
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator)

Epoch 1/10
  8/158 [>.............................] - ETA: 16:43:23 - loss: 3.7545 - accuracy: 0.1680

KeyboardInterrupt: 